# Notebook 05: Identity & OAuth Integration

## Learning Objectives
- Set up Google Drive OAuth integration with AgentCore Identity
- Implement secure credential management with session binding
- Deploy travel agent with OAuth2 authentication
- Create identity-aware travel tools for Google Drive
- Test complete OAuth2 3-legged flow

## Prerequisites
- Completed Notebook 03 (Gateway Integration) - Required for Cognito pool
- Completed Notebook 04 (Memory Implementation)
- Google Cloud Console account
- Google Drive API enabled
- OAuth 2.0 credentials configured
- Docker running
This notebook runs TypeScript on the Deno kernel. Pick the **Deno** kernel in the top right.


## Step 1: Connect to your AWS environment

In [ ]:
Deno.env.set("AWS_REGION", "us-east-1");

// APPROACH A: Use credentials
// Deno.env.set("AWS_ACCESS_KEY_ID", "your_access_key");
// Deno.env.set("AWS_SECRET_ACCESS_KEY", "your_secret_key");
// Deno.env.set("AWS_SESSION_TOKEN", "your_session_token");

// APPROACH B: Use AWS SSO profile
// Deno.env.set("AWS_PROFILE", "your_profile");

console.log("\u2705 AWS Profile set. Please restart kernel and run all cells.");

In [ ]:
import { IdentityClient, Runtime } from "../toolkit/mod.ts";
import { loadEnv, state, writeFile } from "../shared/notebook.ts";

await loadEnv();
const region = Deno.env.get("AWS_REGION") ?? "us-east-1";

console.log("\u2705 Identity imports successful");

## Step 2: Create Test User in Existing Cognito Pool

In [ ]:
// Load existing Cognito config from notebook 03
import {
  CognitoIdentityProviderClient,
  AdminCreateUserCommand,
  AdminSetUserPasswordCommand,
} from "@aws-sdk/client-cognito-identity-provider";
import { ensureUserPasswordAuth, loadCognitoConfig } from "../backend/cognito_config.ts";

const GATEWAY_NAME = "TravelMateGateway";

console.log("\ud83d\udd10 Loading existing Cognito configuration...");
const cognitoResult = await loadCognitoConfig(GATEWAY_NAME);

if (!cognitoResult) {
  console.log("\u274c No existing Cognito config found. Please run notebook 03 first.");
  throw new Error("Missing Cognito configuration from notebook 03");
}

// Create test user in existing pool
console.log("\ud83d\udc64 Creating test user in existing Cognito pool...");
const cognitoClient = new CognitoIdentityProviderClient({ region });

// Get pool ID directly from client_info
const poolId = cognitoResult.client_info.user_pool_id;

try {
  // Create test user
  await cognitoClient.send(new AdminCreateUserCommand({
    UserPoolId: poolId,
    Username: "testuser",
    TemporaryPassword: "Temp123!",
    MessageAction: "SUPPRESS",
  }));

  // Set permanent password
  await cognitoClient.send(new AdminSetUserPasswordCommand({
    UserPoolId: poolId,
    Username: "testuser",
    Password: "MyPassword123!",
    Permanent: true,
  }));
  console.log("\u2705 Test user created: testuser / MyPassword123!");
} catch (error) {
  if ((error as Error).name !== "UsernameExistsException") throw error;
  console.log("\u2705 Test user already exists: testuser / MyPassword123!");
}

// Ensure USER_PASSWORD_AUTH is enabled
await ensureUserPasswordAuth(cognitoResult.client_info.client_id, poolId, region);

console.log("\u2705 Cognito configuration loaded");
console.log(`Client ID: ${cognitoResult.client_info.client_id}`);
console.log(`Scope: ${cognitoResult.client_info.scope}`);

## Step 3: Google OAuth Setup Guide

### Configure Google Drive API Access

Follow these steps to set up Google Drive integration:

#### 1. Create Google Cloud Project
- Go to [Google Cloud Console](https://console.cloud.google.com/)
- Create a new project or select existing one

#### 2. Enable Google Drive API
- Navigate to **APIs & Services > Library**
- Search for "Google Drive API"
- Click **Enable**

#### 3. Configure OAuth Consent Screen
- Go to **APIs & Services > OAuth consent screen**
- Choose **External** user type
- Fill required fields:
  - App name: "AI Travel Companion"
  - User support email: Your email
  - Developer contact: Your email
- Add your email as a test user

#### 4. Create OAuth 2.0 Credentials
- Go to **APIs & Services > Credentials**
- Click **Create Credentials > OAuth client ID**
- Choose **Web application**
- Name: "Travel Companion OAuth"
- **Important**: We'll add the callback URL after creating the provider

#### 5. Get Client ID and Secret
- Copy the **Client ID** and **Client Secret**

In [ ]:
// Google OAuth credentials (or put them in the .env file at the repo root)
// Deno.env.set("GOOGLE_CLIENT_ID", "");
// Deno.env.set("GOOGLE_CLIENT_SECRET", "");

const googleClientId = Deno.env.get("GOOGLE_CLIENT_ID");
const googleClientSecret = Deno.env.get("GOOGLE_CLIENT_SECRET");

if (!googleClientId || !googleClientSecret) {
  throw new Error("Set GOOGLE_CLIENT_ID and GOOGLE_CLIENT_SECRET (see the steps above)");
}

console.log("\u2705 Google OAuth credentials configured");
console.log(`Client ID: ${googleClientId.slice(0, 20)}...`);

## Step 4: Create OAuth2 Credential Provider

In [ ]:
// Configuration
const PROVIDER_NAME = "google-drive-provider";
const identityClient = new IdentityClient({ region });

console.log("\ud83d\udd10 Creating Google Drive OAuth2 Credential Provider...");

// Create the provider if needed, otherwise recover the existing one so reruns work.
let googleProvider;
try {
  googleProvider = await identityClient.createOauth2CredentialProvider({
    name: PROVIDER_NAME,
    credentialProviderVendor: "GoogleOauth2",
    oauth2ProviderConfigInput: {
      googleOauth2ProviderConfig: {
        clientId: googleClientId,
        clientSecret: googleClientSecret,
      },
    },
  });
  console.log(`\u2705 Created new OAuth2 Provider: ${googleProvider.name}`);
  console.log(`\ud83d\udccb AgentCore Callback URL: ${googleProvider.callbackUrl}`);
  console.log("\n\u26a0\ufe0f IMPORTANT: Add this callback URL to your Google OAuth2 client configuration!");
} catch (error) {
  if (!String(error).toLowerCase().includes("already exists")) {
    console.log(`\u274c Error creating OAuth2 provider: ${error}`);
    throw error;
  }
  console.log(`\u2705 Provider '${PROVIDER_NAME}' already exists - retrieving existing configuration...`);
  googleProvider = await identityClient.getOauth2CredentialProvider({ name: PROVIDER_NAME });
  console.log(`\u2705 Retrieved existing OAuth2 Provider: ${googleProvider.name}`);
  console.log(`\ud83d\udccb Provider ARN: ${googleProvider.credentialProviderArn}`);
  console.log(`\ud83d\udccb AgentCore Callback URL: ${googleProvider.callbackUrl}`);
}

## Step 5: Create OAuth2 Callback Server

In [ ]:
await writeFile("../backend/identity/runtime/oauth2_callback_server.ts", `#!/usr/bin/env -S deno run -A
/**
 * OAuth2 Callback Server for Google Drive Integration
 * Handles OAuth2 3-legged authentication flow with AgentCore Identity.
 *
 * Replaces \`oauth2_callback_server.py\`: \`Deno.serve\` in place of FastAPI + uvicorn, and the
 * course's own \`IdentityClient\` in place of \`bedrock_agentcore.services.identity\`.
 *
 * This runs on the learner's own machine, not in AgentCore Runtime: Google redirects the browser
 * here after consent, and this server hands the session back to AgentCore Identity.
 */
import { IdentityClient, type UserIdentifierInput } from "../../../toolkit/mod.ts";

// Configuration constants
export const OAUTH2_CALLBACK_SERVER_PORT = 9090;
export const PING_ENDPOINT = "/ping";
export const OAUTH2_CALLBACK_ENDPOINT = "/oauth2/callback";
export const USER_IDENTIFIER_ENDPOINT = "/userIdentifier/token";

const SUCCESS_HTML = \`
<!DOCTYPE html>
<html>
<head>
    <title>OAuth2 Success</title>
    <style>
        body {
            margin: 0; padding: 0; height: 100vh;
            display: flex; justify-content: center; align-items: center;
            font-family: Arial, sans-serif; background-color: #f5f5f5;
        }
        .container {
            text-align: center; padding: 2rem; background-color: white;
            border-radius: 8px; box-shadow: 0 2px 10px rgba(0, 0, 0, 0.1);
        }
        h1 { color: #28a745; margin: 0; }
    </style>
</head>
<body>
    <div class="container">
        <h1>✅ Google Drive OAuth2 Authorization Successful!</h1>
        <p>You can now close this window and return to the application.</p>
    </div>
</body>
</html>
\`;

export class OAuth2CallbackServer {
  #identityClient: IdentityClient;
  #userTokenIdentifier: UserIdentifierInput | null = null;

  constructor(region: string) {
    this.#identityClient = new IdentityClient({ region });
  }

  /** The request handler, kept separate from \`serve()\` so it can be unit-tested. */
  handler = async (request: Request): Promise<Response> => {
    const url = new URL(request.url);

    if (request.method === "POST" && url.pathname === USER_IDENTIFIER_ENDPOINT) {
      this.#userTokenIdentifier = await request.json() as UserIdentifierInput;
      return new Response(null, { status: 200 });
    }

    if (request.method === "GET" && url.pathname === PING_ENDPOINT) {
      return Response.json({ status: "success" });
    }

    if (request.method === "GET" && url.pathname === OAUTH2_CALLBACK_ENDPOINT) {
      const sessionId = url.searchParams.get("session_id");
      if (!sessionId) {
        return Response.json({ detail: "Missing session_id query parameter" }, { status: 400 });
      }
      if (!this.#userTokenIdentifier) {
        console.error("No configured user token identifier");
        return Response.json({ detail: "Internal Server Error" }, { status: 500 });
      }

      await this.#identityClient.completeResourceTokenAuth({
        sessionUri: sessionId,
        userIdentifier: this.#userTokenIdentifier,
      });

      return new Response(SUCCESS_HTML, {
        status: 200,
        headers: { "content-type": "text/html; charset=utf-8" },
      });
    }

    return new Response("Not Found", { status: 404 });
  };

  serve(): Deno.HttpServer {
    return Deno.serve(
      { hostname: "127.0.0.1", port: OAUTH2_CALLBACK_SERVER_PORT },
      this.handler,
    );
  }
}

export function getOauth2CallbackUrl(): string {
  return \`http://localhost:\${OAUTH2_CALLBACK_SERVER_PORT}\${OAUTH2_CALLBACK_ENDPOINT}\`;
}

export async function storeTokenInOauth2CallbackServer(userTokenValue: string): Promise<void> {
  if (!userTokenValue) return;
  await fetch(\`http://localhost:\${OAUTH2_CALLBACK_SERVER_PORT}\${USER_IDENTIFIER_ENDPOINT}\`, {
    method: "POST",
    headers: { "content-type": "application/json" },
    body: JSON.stringify({ userToken: userTokenValue }),
    signal: AbortSignal.timeout(2000),
  });
}

export async function waitForOauth2ServerToBeReady(durationSeconds = 40): Promise<boolean> {
  const deadline = Date.now() + durationSeconds * 1000;
  while (Date.now() < deadline) {
    try {
      const response = await fetch(
        \`http://localhost:\${OAUTH2_CALLBACK_SERVER_PORT}\${PING_ENDPOINT}\`,
        { signal: AbortSignal.timeout(2000) },
      );
      if (response.ok) {
        await response.body?.cancel();
        return true;
      }
    } catch {
      // server not up yet
    }
    await new Promise((resolve) => setTimeout(resolve, 2000));
  }
  return false;
}

if (import.meta.main) {
  // Python used argparse: \`-r/--region\`, required.
  const args = Deno.args;
  const regionIndex = args.findIndex((a) => a === "-r" || a === "--region");
  const region = regionIndex >= 0 ? args[regionIndex + 1] : Deno.env.get("AWS_REGION");
  if (!region) {
    console.error("Usage: oauth2_callback_server.ts --region <AWS Region>");
    Deno.exit(2);
  }

  console.log(\`OAuth2 callback server listening on \${getOauth2CallbackUrl()}\`);
  new OAuth2CallbackServer(region).serve();
}
`);

## Step 6: Create Travel Agent with Google Drive Integration

In [ ]:
await writeFile("../backend/identity/runtime/travel_agent_google_drive.ts", `#!/usr/bin/env -S deno run -A
/**
 * Travel Agent with Google Drive Integration
 * Uses AgentCore Identity for OAuth2 authentication with Google Drive.
 *
 * Replaces \`travel_agent_google_drive.py\`. The Python \`@requires_access_token(...)\` decorator
 * becomes \`withAccessToken({...})(fn)\` from \`bedrock-agentcore/identity\`, which injects the token
 * as the wrapped function's last argument. Google's client comes from \`npm:googleapis\`.
 */
import { Agent, tool } from "@strands-agents/sdk";
import { BedrockAgentCoreApp } from "bedrock-agentcore/runtime";
import { withAccessToken } from "bedrock-agentcore/identity";
import { google } from "googleapis";
import { z } from "zod";
import { getOauth2CallbackUrl } from "./oauth2_callback_server.ts";

// Environment configuration
Deno.env.set("STRANDS_OTEL_ENABLE_CONSOLE_EXPORT", "true");
Deno.env.set("OTEL_PYTHON_EXCLUDED_URLS", "/ping,/invocations");

// Google Drive API scope
const SCOPES = ["https://www.googleapis.com/auth/drive.file"];

// Module-level access token, set once the OAuth2 flow completes
let googleAccessToken: string | null = null;

/** Saves a travel itinerary to Google Drive as a text file. */
const saveItineraryToDrive = tool({
  name: "save_itinerary_to_drive",
  description: "Saves a travel itinerary to Google Drive as a text file",
  inputSchema: z.object({
    destination: z.string().describe("Travel destination name"),
    itinerary_content: z.string().describe("The itinerary content to save"),
  }),
  callback: async ({ destination, itinerary_content }) => {
    if (!googleAccessToken) {
      return JSON.stringify({
        message:
          "Google Drive authentication is required. Please wait while we set up the authorization.",
        success: false,
      });
    }

    try {
      // Create credentials from access token
      const auth = new google.auth.OAuth2();
      auth.setCredentials({ access_token: googleAccessToken, scope: SCOPES.join(" ") });
      const service = google.drive({ version: "v3", auth });

      // Create filename
      const date = new Date().toISOString().slice(0, 10).replace(/-/g, "");
      const filename = \`\${destination.toLowerCase().replace(/ /g, "_")}_itinerary_\${date}.txt\`;

      // Upload file
      const file = await service.files.create({
        requestBody: { name: filename },
        media: { mimeType: "text/plain", body: itinerary_content },
        fields: "id,name,webViewLink",
      });

      return JSON.stringify({
        success: true,
        message: \`✅ Itinerary saved to Google Drive: \${file.data.name}\`,
        file_id: file.data.id,
        view_link: file.data.webViewLink,
      });
    } catch (error) {
      return JSON.stringify({
        success: false,
        error: \`Error saving to Google Drive: \${error}\`,
      });
    }
  },
});

// Initialize the agent
const agent = new Agent({
  model: "us.anthropic.claude-sonnet-4-6",
  tools: [saveItineraryToDrive],
  systemPrompt: \`
You are a helpful travel planning assistant with the ability to save itineraries to Google Drive.
When users ask you to create travel plans, generate detailed itineraries and offer to save them to Google Drive.
Always format itineraries clearly with day-by-day breakdowns, times, and activities.
\`,
});

// Initialize app
const app = new BedrockAgentCoreApp({
  invocationHandler: {
    requestSchema: z.object({
      prompt: z.string().default(
        "Hello! I'm your travel planning assistant. How can I help you plan your next trip?",
      ),
    }),
    process: async function* ({ prompt }) {
      yield* agentTask(prompt);
    },
  },
});

/** Handle authorization URL callback: the learner opens this URL to grant Drive access. */
async function onAuthUrl(url: string): Promise<void> {
  console.log(\`Authorization url: \${url}\`);
  pendingAuthUrl = url;
  await Promise.resolve();
}

let pendingAuthUrl: string | null = null;

/**
 * Get Google Drive access token.
 *
 * Python wrapped this with \`@requires_access_token(...)\`; here \`withAccessToken\` injects the token
 * as the last argument. Outside a runtime request it throws, because there is no workload identity.
 */
const getGoogleDriveToken = withAccessToken({
  providerName: "google-drive-provider",
  scopes: SCOPES,
  authFlow: "USER_FEDERATION",
  onAuthUrl,
  forceAuthentication: true,
  callbackUrl: getOauth2CallbackUrl(),
})(async (accessToken: string): Promise<string> => {
  googleAccessToken = accessToken;
  return await Promise.resolve(accessToken);
});

/**
 * Execute the agent task with authentication handling.
 *
 * Python pushed progress onto an asyncio queue and streamed it; an async generator says the same
 * thing directly, so the StreamingQueue class has no counterpart here.
 */
async function* agentTask(userMessage: string): AsyncGenerator<string> {
  try {
    yield "Begin agent execution";

    // Call the agent first to see if it needs authentication
    let response = await agent.invoke(userMessage);
    const responseText = response.toString();

    // Check if the response indicates authentication is required
    const authKeywords = [
      "authentication",
      "authorize",
      "authorization",
      "auth",
      "sign in",
      "login",
      "access",
      "permission",
      "credential",
      "need authentication",
      "requires authentication",
    ];
    const needsAuth = authKeywords.some((k) => responseText.toLowerCase().includes(k));

    if (needsAuth) {
      yield "Authentication required for Google Drive access. Starting authorization flow...";

      // Trigger the 3LO authentication flow
      try {
        googleAccessToken = await getGoogleDriveToken();
        if (pendingAuthUrl) yield \`Authorization url: \${pendingAuthUrl}\`;
        yield "Authentication successful! Retrying your request...";

        // Retry the agent call now that we have authentication
        response = await agent.invoke(userMessage);
      } catch (authError) {
        console.log(\`auth_error: \${authError}\`);
        yield \`Authentication failed: \${authError}\`;
      }
    }

    yield response.toString();
    yield "End agent execution";
  } catch (error) {
    yield \`Error: \${error}\`;
  }
}

if (import.meta.main) {
  app.run();
}
`);

## Step 7: Configure AgentCore Runtime Deployment

In [ ]:
await writeFile("../backend/identity/runtime/deno.json", `{
  "nodeModulesDir": "auto",
  "compilerOptions": {
    "strict": true
  },
  "imports": {
    "@strands-agents/sdk": "npm:@strands-agents/sdk@1.18.0",
    "bedrock-agentcore/": "npm:/bedrock-agentcore@0.4.4/",
    "zod": "npm:zod@4.6.5",
    "googleapis": "npm:googleapis@181.0.0",
    "@opentelemetry/api": "npm:@opentelemetry/api@1.9.1",
    "@opentelemetry/api-logs": "npm:@opentelemetry/api-logs@0.219.0",
    "@opentelemetry/context-async-hooks": "npm:@opentelemetry/context-async-hooks@2.8.0",
    "@opentelemetry/core": "npm:@opentelemetry/core@2.8.0",
    "@opentelemetry/otlp-transformer": "npm:@opentelemetry/otlp-transformer@0.219.0",
    "@opentelemetry/resources": "npm:@opentelemetry/resources@2.8.0",
    "@opentelemetry/sdk-logs": "npm:@opentelemetry/sdk-logs@0.219.0",
    "@opentelemetry/sdk-trace-base": "npm:@opentelemetry/sdk-trace-base@2.8.0",
    "@smithy/protocol-http": "npm:@smithy/protocol-http@5.6.2",
    "@smithy/signature-v4": "npm:@smithy/signature-v4@5.7.3",
    "@aws-crypto/sha256-js": "npm:@aws-crypto/sha256-js@5.2.0",
    "@aws-sdk/credential-provider-node": "npm:@aws-sdk/credential-provider-node@3.972.83",
    "@aws-sdk/client-bedrock-agentcore": "npm:@aws-sdk/client-bedrock-agentcore@3.1136.0",
    "@aws-sdk/client-bedrock-agentcore-control": "npm:@aws-sdk/client-bedrock-agentcore-control@3.1136.0",
    "@std/path": "jsr:@std/path@1.1.6"
  }
}
`);

In [ ]:
console.log("\ud83d\ude80 Configuring AgentCore Runtime deployment...");

const discoveryUrl = cognitoResult.authorizer_config.customJWTAuthorizer?.discoveryUrl;
const clientId = cognitoResult.client_info.client_id;

const agentcoreRuntime = new Runtime();

const response = await agentcoreRuntime.configure({
  entrypoint: "travel_agent_google_drive.ts",
  autoCreateExecutionRole: true,
  autoCreateEcr: true,
  requirementsFile: "deno.json",
  region,
  agentName: "travel_agent_google_drive",
  sourceDir: "../backend/identity/runtime",
  ...(discoveryUrl && clientId
    ? {
      authorizerConfiguration: {
        customJWTAuthorizer: { discoveryUrl, allowedClients: [clientId] },
      },
    }
    : {}),
});

console.log("\u2705 Runtime configuration completed");
console.log(response);

## Step 8: Start OAuth2 Callback Server

Before testing the OAuth2 flow, you need to start the callback server in a separate terminal.
It runs on your own machine: Google redirects your browser to it after you grant access.

In [ ]:
console.log("\ud83d\udca1 OAuth2 callback server setup instructions:");
console.log("");
console.log("1. Open a separate terminal");
console.log("2. Navigate to the project root directory");
console.log("3. Run the following command:");
console.log("");
console.log("   AWS_REGION=us-east-1 deno task oauth:server --region us-east-1");
console.log("");
console.log("4. Leave it running while you test the agent below");

## Step 9: Deploy Agent to AgentCore Runtime

In [ ]:
console.log("\ud83d\ude80 Deploying agent to AgentCore Runtime...");

// Deploy the agent
const launchResult = await agentcoreRuntime.launch();
console.log(`\u2705 Agent deployed: ${launchResult.agentId}`);

In [ ]:
// Update workload identity with OAuth2 callback URL
import { getOauth2CallbackUrl } from "../backend/identity/runtime/oauth2_callback_server.ts";

const workloadName = launchResult.agentId;
const workloadIdentity = await identityClient.getWorkloadIdentity({ name: workloadName });
const allowedUrls = workloadIdentity.allowedResourceOauth2ReturnUrls ?? [];
const oauth2CallbackUrl = getOauth2CallbackUrl();

console.log(`\ud83d\udd17 Updating workload ${workloadName} with callback URL: ${oauth2CallbackUrl}`);

// Register the callback URL
await identityClient.updateWorkloadIdentity({
  name: workloadName,
  allowedResourceOauth2ReturnUrls: [...allowedUrls, oauth2CallbackUrl],
});

console.log("\u2705 Workload identity updated with OAuth2 callback URL");

## Step 10: Verify Deployment Status

In [ ]:
console.log("\u23f3 Waiting for AgentCore Runtime to be ready...");

let statusResponse = await agentcoreRuntime.status();
let status = (statusResponse.endpoint as { status?: string })?.status ?? "UNKNOWN";
const endStatus = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"];

while (!endStatus.includes(status)) {
  console.log(`Status: ${status}`);
  await new Promise((resolve) => setTimeout(resolve, 15_000));
  statusResponse = await agentcoreRuntime.status();
  status = (statusResponse.endpoint as { status?: string })?.status ?? "UNKNOWN";
}

console.log(`\ud83c\udf89 Final Status: ${status}`);

## Step 11: Test the Travel Agent with Google Drive Integration

In [ ]:
import { reauthenticateUser } from "../backend/auth_utils.ts";
import { storeTokenInOauth2CallbackServer, waitForOauth2ServerToBeReady } from "../backend/identity/runtime/oauth2_callback_server.ts";

console.log("\ud83e\uddea Testing Travel Agent with Google Drive Integration...");

// Get bearer token for authentication using test user
const bearerToken = await reauthenticateUser(cognitoResult.client_info.client_id);

if (!bearerToken) {
  console.log("\u274c Failed to get access token");
} else {
  console.log(`\u2705 Got access token: ${bearerToken.slice(0, 20)}...`);

  const testPrompt = `
    Create a 3-day travel itinerary for Tokyo, Japan including:
    - Traditional temples and gardens
    - Modern attractions like Tokyo Skytree
    - Food experiences and restaurants
    - Shopping districts

    Please save this itinerary to my Google Drive without asking again if you're allow to.
    `;

  console.log("\ud83e\udd16 Invoking travel agent...");
  try {
    if (!(await waitForOauth2ServerToBeReady(5))) {
      console.log("\u26a0\ufe0f Callback server not reachable on port 9090 - start it (Step 8) for the OAuth2 flow");
    }
    await storeTokenInOauth2CallbackServer(bearerToken);

    const invokeResponse = await agentcoreRuntime.invoke({ prompt: testPrompt }, { bearerToken });

    console.log("\ud83d\udcdd Agent Response:");
    console.log(invokeResponse);
    console.log("\n\ud83d\udc46 If the agent printed an 'Authorization url', open it in your browser to grant Drive access.");
  } catch (error) {
    console.log(`\u274c Error during testing: ${error}`);
    console.log("\n\ud83d\udca1 Note: To test the OAuth2 flow, start the callback server manually:");
    console.log("   deno task oauth:server --region us-east-1");
  }
}

In [ ]:
// Save identity configuration for notebook 08
const identityConfig = {
  oauth2_provider: {
    name: PROVIDER_NAME,
    // The API returns an ARN, not an id; Python read `credentialProviderId` and always saved null.
    provider_arn: googleProvider.credentialProviderArn ?? null,
    callback_url: googleProvider.callbackUrl ?? null,
    vendor: "GoogleOauth2",
  },
  google_oauth: { client_id: googleClientId },
  agent_runtime: {
    name: "travel_agent_google_drive",
    entrypoint: "travel_agent_google_drive.ts",
    oauth_callback_server_port: 9090,
  },
  cognito_integration: {
    user_pool_id: cognitoResult.client_info.user_pool_id,
    client_id: cognitoResult.client_info.client_id,
    discovery_url: cognitoResult.authorizer_config.customJWTAuthorizer?.discoveryUrl ?? null,
  },
  region,
};

// Save to environments directory
await writeFile("environments/identity_info.json", `${JSON.stringify(identityConfig, null, 2)}\n`);
await state.set("identity_info", identityConfig);

console.log("\u2705 Identity configuration saved to environments/identity_info.json");
console.log(`\ud83d\udccb OAuth2 Provider: ${PROVIDER_NAME}`);
console.log(`\ud83d\udccb Callback URL: ${googleProvider.callbackUrl}`);

## Step 12: Integration Summary

### What We've Accomplished

1. **Complete OAuth2 Flow**: Implemented proper 3-legged OAuth with session binding
2. **Cognito Integration**: Set up inbound authentication with Cognito
3. **AgentCore Runtime**: Deployed travel agent to runtime with proper authorization
4. **Google Drive Integration**: Created tools to save travel itineraries to Google Drive
5. **Session Binding**: Implemented secure OAuth2 session binding with callback server
6. **Workload Identity**: Properly configured workload identity with callback URLs

### Key Features

- **Secure Authentication**: Uses AgentCore Identity for OAuth2 management
- **Session Binding**: Prevents OAuth token hijacking with proper user validation
- **Travel Planning**: AI-powered travel itinerary generation
- **Google Drive Storage**: Automatic saving of itineraries to user's Google Drive
- **Production Ready**: Deployed to AgentCore Runtime with proper authorization

### Next Steps

- **Google Console**: Ensure the AgentCore callback URL is added to your Google OAuth2 client
- **Testing**: Use the test credentials (testuser / MyPassword123!) for Cognito authentication
- **Expansion**: Add more travel tools like flight booking, hotel reservations, etc.
- **Production**: Deploy with proper HTTPS endpoints for production use

### Usage Flow

1. User authenticates with Cognito
2. User requests travel planning with Google Drive save
3. Agent triggers OAuth2 flow for Google Drive access
4. User authorizes Google Drive access in browser
5. Agent saves itinerary to Google Drive
6. User receives confirmation with Google Drive link

The travel agent is now fully integrated with Google Drive using proper AgentCore Identity OAuth2 flow!